<a href="https://colab.research.google.com/github/vishalmysore/AI/blob/main/foundation_instruct_chat_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foundation vs. Instruct vs. Chat Models
### A hands-on tutorial you can run in Google Colab

Three models. One question. Three very different answers.

When people say "LLM" they often mean very different things. In this notebook we ask the **exact same question** to three flavors of the *same* tiny model family and watch how the answer changes:

| # | Model type | What it was trained to do | How you talk to it |
|---|------------|---------------------------|--------------------|
| 1 | **Foundation (base)** | Predict the next token on raw internet text | Plain string, no structure |
| 2 | **Instruct** | Follow a single instruction | A formatted prompt (chat template) |
| 3 | **Chat** | Hold a multi-turn conversation | A list of role-tagged messages |

We use the **SmolLM2-135M** family from Hugging Face. At 135M parameters these are small enough to run on a **free Colab CPU** in a couple of minutes — no GPU required.

> **Tip:** Run the cells top to bottom (`Shift+Enter`). The first run downloads ~3 model files, so give it a minute.

## 0. Setup

Install the dependencies. On Colab `torch` is usually pre-installed, so this is quick.

In [1]:
%pip install -q -U transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 907.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59

In [1]:
import torch
from transformers import pipeline, AutoTokenizer

# Silence the noisy generation warnings so the output stays readable
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

# The single question we ask all three models.
test_query = "What is the capital of France?"

print("=" * 60)
print(f"THE BENCHMARK QUERY: '{test_query}'")
print("=" * 60)

THE BENCHMARK QUERY: 'What is the capital of France?'


## 1. The Foundation (Base) Model

A **foundation model** is trained on one objective only: *given some text, predict the next word*. It has read a huge slice of the internet, so it "knows" a lot — but **nobody ever taught it to answer questions or be helpful.**

So when we hand it `"What is the capital of France?"` it does not think *"I should answer this."* It thinks *"What text usually **follows** a sentence like this?"* — which on the internet is often **more questions**, a quiz, or a rambling continuation.

Watch: we pass the raw string straight in, with no formatting at all.

In [2]:
print("[Loading 1/3: Foundation Model...]")
base_id = "HuggingFaceTB/SmolLM2-135M"  # NOTE: no '-Instruct' suffix = the raw base model

base_pipe = pipeline("text-generation", model=base_id, clean_up_tokenization_spaces=False)

# do_sample=False -> greedy/deterministic so you get the same result we describe below
base_raw_out = base_pipe(test_query, max_new_tokens=30, do_sample=False)

print("\n--- 1. FOUNDATION MODEL OUTPUT ---")
print(base_raw_out[0]['generated_text'])
print("-" * 35)

[Loading 1/3: Foundation Model...]


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]


--- 1. FOUNDATION MODEL OUTPUT ---
What is the capital of France?

The capital of France is Paris.

What is the capital of France?

Paris is the capital of France.

What
-----------------------------------


**What you just saw:** the base model echoes your question back and *continues the document* — often with more trivia questions or a tangent. It may even contain the right answer somewhere, but it was never optimized to **stop and answer you**. This is the raw material that everything else is built on.

👉 *Key idea:* a foundation model is a **text completer**, not an **assistant**.

## 2. The Instruct Model

An **instruct model** starts from that same base model and is then **fine-tuned on (instruction → response) pairs**. After this training it understands a new contract: *"when the user asks for something, do it."*

But there's a catch — the model only behaves if you wrap your text in the **exact special format it was trained on**. That format uses control tokens like `<|im_start|>user ... <|im_end|>`. We don't type those by hand; the tokenizer's **chat template** builds them for us.

Below we print the formatted prompt first, so you can *see* the hidden scaffolding the model actually receives.

In [3]:
print("[Loading 2/3: Instruct Model...]")
instruct_id = "HuggingFaceTB/SmolLM2-135M-Instruct"  # same family, instruction-tuned

instruct_pipe = pipeline("text-generation", model=instruct_id, clean_up_tokenization_spaces=False)
tokenizer = AutoTokenizer.from_pretrained(instruct_id)

# 1. Describe the turn as structured data
instruct_messages = [{"role": "user", "content": test_query}]

# 2. The chat template renders it into the exact special-token string the model expects
formatted_prompt = tokenizer.apply_chat_template(
    instruct_messages,
    tokenize=False,
    add_generation_prompt=True,  # adds the '<|im_start|>assistant' cue so the model knows it's its turn
)

print("\n--- THE ACTUAL PROMPT THE MODEL SEES ---")
print(formatted_prompt)
print("-" * 40)

instruct_out = instruct_pipe(formatted_prompt, max_new_tokens=30, do_sample=False)
# Slice off the prompt so we only keep what the model newly generated
instruct_response = instruct_out[0]['generated_text'][len(formatted_prompt):]

print("\n--- 2. INSTRUCT MODEL OUTPUT ---")
print(instruct_response.strip())
print("-" * 35)

[Loading 2/3: Instruct Model...]


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]


--- THE ACTUAL PROMPT THE MODEL SEES ---
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant

----------------------------------------

--- 2. INSTRUCT MODEL OUTPUT ---
The capital of France is Paris.
-----------------------------------


**What you just saw:** the *same size* model now gives a clean, direct answer ("The capital of France is Paris."). The weights changed only a little — what changed a lot is that it learned the **instruction-following contract**, and we spoke to it using the **chat template** it expects.

👉 *Key idea:* an instruct model is a base model **+ instruction tuning + a required prompt format**.

## 3. The Chat Model

A **chat model** is an instruct model used the way real assistants work: you keep a **running list of messages** with `role`s (`user`, `assistant`, `system`), and the model continues the conversation. In practice the *weights* are often the very same instruct checkpoint — what's new is the **interface and the multi-turn memory**.

The `pipeline` accepts a list of messages directly: it applies the chat template for you and returns the **updated conversation** with the assistant's reply appended.

In [4]:
print("[Loading 3/3: Chat Model...]")
# Reuse the instruct checkpoint - 'chat' here is about how we *use* it, not a different download
chat_pipe = pipeline("text-generation", model=instruct_id, clean_up_tokenization_spaces=False)

# A conversation is just a list of role-tagged messages
chat_history = [
    {"role": "user", "content": test_query},
]

# Pass the list straight in - the pipeline templates it and appends the reply
chat_out = chat_pipe(chat_history, max_new_tokens=30, do_sample=False)

print("\n--- 3. CHAT MODEL OUTPUT ---")
print(f"Assistant: {chat_out[0]['generated_text'][-1]['content'].strip()}")
print("-" * 35)

[Loading 3/3: Chat Model...]


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


--- 3. CHAT MODEL OUTPUT ---
Assistant: The capital of France is Paris.
-----------------------------------


### The real superpower: memory across turns

A single answer looks just like the instruct model. The difference shows up when the conversation **continues** — a chat model uses the earlier turns as context. Let's ask a vague follow-up that only makes sense if it *remembers* we were talking about Paris.

In [5]:
# Append the assistant's reply, then ask a follow-up that depends on memory
conversation = chat_out[0]['generated_text']  # already contains user + assistant turns
conversation.append({"role": "user", "content": "And what is a famous landmark there?"})

follow_up = chat_pipe(conversation, max_new_tokens=40, do_sample=False)

print("\n--- MULTI-TURN CONVERSATION ---")
for turn in follow_up[0]['generated_text']:
    print(f"{turn['role'].upper():>9}: {turn['content'].strip()}")
print("-" * 35)


--- MULTI-TURN CONVERSATION ---
     USER: What is the capital of France?
ASSISTANT: The capital of France is Paris.
     USER: And what is a famous landmark there?
ASSISTANT: One of the most famous landmarks in Paris is the Eiffel Tower, also known as the Champ de Mars. It's a towering iron latticework structure that stands over 320 meters tall and
-----------------------------------


**What you just saw:** the word "there" has no meaning on its own. Because we passed the **whole history**, the chat model resolves "there" to **Paris** and names a landmark. That carried-over context is what makes it feel like a conversation rather than a one-shot Q&A.

👉 *Key idea:* a chat model is an instruct model **driven through a multi-turn message list** so it can use prior turns as context.

> ⚠️ A 135M model is tiny — it may occasionally get a fact wrong or ramble. That's expected at this size; the *behavioral* difference between the three modes is the point, not benchmark accuracy.

## Recap

Same model family, one question, three behaviors:

| Model | Trained to… | You give it… | Typical reply to *"What is the capital of France?"* |
|-------|-------------|--------------|------------------------------------------------------|
| **Foundation** | continue text | a raw string | echoes / continues the document, may not answer |
| **Instruct** | follow one instruction | a chat-templated string | a direct answer: *"The capital of France is Paris."* |
| **Chat** | converse over many turns | a list of messages | a direct answer **+ remembers context** for follow-ups |

### The one-sentence takeaways
- **Foundation** models *complete text*. Powerful, but not helpful out of the box.
- **Instruct** models add *instruction tuning* — and they need the *right prompt format* to shine.
- **Chat** models wrap that in a *multi-turn message interface* so context carries across turns.

### Try it yourself
- Change `test_query` to something open-ended like `"Write a haiku about the sea."` and rerun.
- Set `do_sample=True` (and add `temperature=0.7`) to see more varied, creative outputs.
- Swap in a bigger sibling such as `HuggingFaceTB/SmolLM2-360M-Instruct` and compare quality.

*Happy experimenting!* 🚀